# Notebook 03 — DML nas tabelas Delta (INSERT / UPDATE / DELETE)

Demonstra as três operações de DML nas tabelas Delta do bucket `bronze`, além de **Time Travel** e **DESCRIBE HISTORY** para auditoria.

**Tabelas utilizadas:** `cliente`, `apolice`, `sinistro`

## 1. Configuração — PYSPARK_SUBMIT_ARGS deve ser definido ANTES de importar pyspark

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

S3A_ENDPOINT = MINIO_ENDPOINT.replace('http://', '').replace('https://', '')

# DEVE ser definido ANTES de importar pyspark
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages '
    'io.delta:delta-spark_2.12:3.2.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4,'
    'com.amazonaws:aws-java-sdk-bundle:1.12.262 '
    'pyspark-shell'
)

print(f'MinIO: {MINIO_ENDPOINT} | Bucket bronze: {BRONZE_BUCKET}')

MinIO: http://localhost:9020 | Bucket bronze: bronze


## 2. SparkSession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from delta import DeltaTable
import datetime

spark = (
    SparkSession.builder
    .appName('dml-delta-bronze')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', f'http://{S3A_ENDPOINT}')
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

def bronze_path(tabela):
    return f's3a://{BRONZE_BUCKET}/{tabela}'

print(f'Spark {spark.version} iniciado!')

26/05/06 23:40:14 WARN Utils: Your hostname, LAPTOP-NGVDQKNB resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 23:40:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/Users/mazuc/OneDrive/%c3%81rea%20de%20Trabalho/SATC/5%c2%b0%20FASE/Engenharia%20de%20Dados/trabalho-2-delta-minio/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thiago/.ivy2/cache
The jars for the packages stored in: /home/thiago/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-76aa0c1d-4991-4998-8bdf-a1ec9fdd69ce;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 720ms :: artifacts dl 37ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-

Spark 3.5.3 iniciado!


## 3. Estado inicial — contagem antes do DML

In [3]:
tabelas_dml = ['cliente', 'apolice', 'sinistro']

print('=== Contagem ANTES do DML ===')
for t in tabelas_dml:
    count = spark.read.format('delta').load(bronze_path(t)).count()
    print(f'  {t:12s}: {count:5d} registros')

=== Contagem ANTES do DML ===


26/05/06 23:40:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/06 23:41:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

  cliente     :    63 registros


  apolice     :    32 registros


[Stage 17:================================================>       (43 + 7) / 50]

  sinistro    :    20 registros


---
## 4. INSERT — Novos clientes e apólices

Adicionamos 3 novos clientes e 2 novas apólices ao bronze.

In [4]:
antes = spark.read.format('delta').load(bronze_path('cliente')).count()
print(f'Clientes antes do INSERT: {antes}')

schema_cliente = StructType([
    StructField('id', IntegerType(), True),
    StructField('nome', StringType(), True),
    StructField('cpf', StringType(), True),
    StructField('data_nascimento', DateType(), True),
    StructField('email', StringType(), True),
    StructField('sexo', StringType(), True),
])

novos_clientes = spark.createDataFrame([
    (61, 'Patricia Oliveira',  '611.611.611-61', datetime.date(1992, 4, 15), 'patricia.oliveira@email.com', 'F'),
    (62, 'Rodrigo Almeida',   '622.622.622-62', datetime.date(1985, 9, 28), 'rodrigo.almeida@email.com',   'M'),
    (63, 'Samantha Ferreira', '633.633.633-63', datetime.date(1998, 1,  7), 'samantha.ferreira@email.com', 'F'),
], schema_cliente)

novos_clientes.write.format('delta').mode('append').save(bronze_path('cliente'))

depois = spark.read.format('delta').load(bronze_path('cliente')).count()
print(f'Clientes depois do INSERT: {depois} (+{depois - antes})')

Clientes antes do INSERT: 63


[Stage 31:======================================================> (49 + 1) / 50]

Clientes depois do INSERT: 66 (+3)


In [5]:
antes_ap = spark.read.format('delta').load(bronze_path('apolice')).count()
print(f'Apólices antes do INSERT: {antes_ap}')

schema_apolice = StructType([
    StructField('id', IntegerType(), True),
    StructField('cliente_id', IntegerType(), True),
    StructField('carro_id', IntegerType(), True),
    StructField('numero_apolice', StringType(), True),
    StructField('data_inicio', DateType(), True),
    StructField('data_fim', DateType(), True),
    StructField('tipo_cobertura', StringType(), True),
    StructField('valor_premio', DoubleType(), True),
    StructField('valor_cobertura', DoubleType(), True),
    StructField('status', StringType(), True),
])

novas_apolices = spark.createDataFrame([
    (31, 61, 1, 'APL-2024-0011', datetime.date(2024, 5,  1), datetime.date(2025, 5,  1), 'Completo',  2950.00, 65000.00, 'ativa'),
    (32, 62, 2, 'APL-2024-0012', datetime.date(2024, 5, 15), datetime.date(2025, 5, 15), 'Terceiros',  890.00, 30000.00, 'ativa'),
], schema_apolice)

novas_apolices.write.format('delta').mode('append').save(bronze_path('apolice'))

depois_ap = spark.read.format('delta').load(bronze_path('apolice')).count()
print(f'Apólices depois do INSERT: {depois_ap} (+{depois_ap - antes_ap})')

Apólices antes do INSERT: 32


[Stage 45:===================================================>    (46 + 4) / 50]

Apólices depois do INSERT: 34 (+2)


---
## 5. UPDATE — Reajuste de prêmios nas apólices

Aplicamos um reajuste de **10%** nas apólices com cobertura 'Completo' e status 'ativa'.

In [6]:
dt_apolice = DeltaTable.forPath(spark, bronze_path('apolice'))

print('=== Apólices Completo/ativa ANTES do UPDATE ===')
spark.read.format('delta').load(bronze_path('apolice')) \
    .filter((col('tipo_cobertura') == 'Completo') & (col('status') == 'ativa')) \
    .select('id', 'numero_apolice', 'tipo_cobertura', 'valor_premio', 'status') \
    .show(5)

dt_apolice.update(
    condition=(col('tipo_cobertura') == 'Completo') & (col('status') == 'ativa'),
    set={'valor_premio': col('valor_premio') * lit(1.10)}
)

print('=== Apólices Completo/ativa APÓS UPDATE (reajuste de 10%) ===')
spark.read.format('delta').load(bronze_path('apolice')) \
    .filter((col('tipo_cobertura') == 'Completo') & (col('status') == 'ativa')) \
    .select('id', 'numero_apolice', 'tipo_cobertura', 'valor_premio', 'status') \
    .show(5)

=== Apólices Completo/ativa ANTES do UPDATE ===


+---+--------------+--------------+------------+------+
| id|numero_apolice|tipo_cobertura|valor_premio|status|
+---+--------------+--------------+------------+------+
|  1| APL-2023-0001|      Completo|      2800.0| ativa|
|  3| APL-2023-0003|      Completo|      4200.0| ativa|
|  4| APL-2023-0004|      Completo|      3100.0| ativa|
|  6| APL-2023-0006|      Completo|      5800.0| ativa|
|  7| APL-2023-0007|      Completo|      3900.0| ativa|
+---+--------------+--------------+------------+------+
only showing top 5 rows



=== Apólices Completo/ativa APÓS UPDATE (reajuste de 10%) ===


[Stage 67:=================================>                      (30 + 8) / 50]

+---+--------------+--------------+------------------+------+
| id|numero_apolice|tipo_cobertura|      valor_premio|status|
+---+--------------+--------------+------------------+------+
|  1| APL-2023-0001|      Completo|3080.0000000000005| ativa|
|  3| APL-2023-0003|      Completo|            4620.0| ativa|
|  4| APL-2023-0004|      Completo|3410.0000000000005| ativa|
|  6| APL-2023-0006|      Completo| 6380.000000000001| ativa|
|  7| APL-2023-0007|      Completo|            4290.0| ativa|
+---+--------------+--------------+------------------+------+
only showing top 5 rows



---
## 6. DELETE — Remover sinistros cancelados

Sinistros com status 'cancelado' são removidos da tabela Delta.

In [7]:
dt_sinistro = DeltaTable.forPath(spark, bronze_path('sinistro'))

antes_s = spark.read.format('delta').load(bronze_path('sinistro')).count()
cancelados = spark.read.format('delta').load(bronze_path('sinistro')) \
    .filter(col('status') == 'cancelado').count()

print(f'Sinistros antes do DELETE: {antes_s}')
print(f'Sinistros com status cancelado: {cancelados}')

dt_sinistro.delete(condition=col('status') == 'cancelado')

depois_s = spark.read.format('delta').load(bronze_path('sinistro')).count()
print(f'Sinistros após DELETE: {depois_s} (-{antes_s - depois_s})')

Sinistros antes do DELETE: 20
Sinistros com status cancelado: 2


[Stage 94:=============================================>          (41 + 8) / 50]

Sinistros após DELETE: 18 (-2)


---
## 7. DESCRIBE HISTORY — Histórico de transações

In [8]:
print('=== Histórico de operações — tabela: cliente ===')
DeltaTable.forPath(spark, bronze_path('cliente')) \
    .history() \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

=== Histórico de operações — tabela: cliente ===
+-------+-------------------+---------+------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                            |
+-------+-------------------+---------+------------------------------------------------------------+
|2      |2026-05-06 23:41:38|WRITE    |{numFiles -> 4, numOutputRows -> 3, numOutputBytes -> 6545} |
|1      |2026-05-06 23:22:51|WRITE    |{numFiles -> 4, numOutputRows -> 3, numOutputBytes -> 6545} |
|0      |2026-05-06 23:06:09|WRITE    |{numFiles -> 1, numOutputRows -> 60, numOutputBytes -> 4861}|
+-------+-------------------+---------+------------------------------------------------------------+



In [9]:
print('=== Histórico de operações — tabela: apolice ===')
DeltaTable.forPath(spark, bronze_path('apolice')) \
    .history() \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

=== Histórico de operações — tabela: apolice ===
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                                 |
+-------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [10]:
print('=== Histórico de operações — tabela: sinistro ===')
DeltaTable.forPath(spark, bronze_path('sinistro')) \
    .history() \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

=== Histórico de operações — tabela: sinistro ===
+-------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                                |
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

---
## 8. Time Travel — Leitura em versão anterior

In [11]:
print('=== Time Travel: tabela apolice na versão 0 (dados originais do SQL Server) ===')
df_v0 = spark.read.format('delta') \
    .option('versionAsOf', 0) \
    .load(bronze_path('apolice'))

print(f'Registros na versão 0: {df_v0.count()}')
df_v0.select('id', 'numero_apolice', 'valor_premio', 'status').show(5)

=== Time Travel: tabela apolice na versão 0 (dados originais do SQL Server) ===


Registros na versão 0: 30


+---+--------------+------------+------+
| id|numero_apolice|valor_premio|status|
+---+--------------+------------+------+
|  1| APL-2023-0001|      2800.0| ativa|
|  2| APL-2023-0002|       980.0| ativa|
|  3| APL-2023-0003|      4200.0| ativa|
|  4| APL-2023-0004|      3100.0| ativa|
|  5| APL-2023-0005|      1200.0| ativa|
+---+--------------+------------+------+
only showing top 5 rows



In [12]:
df_atual = spark.read.format('delta').load(bronze_path('apolice'))

print(f'Apólices na versão 0 (original): {df_v0.count()}')
print(f'Apólices na versão atual:         {df_atual.count()}')

avg_v0    = df_v0.agg({'valor_premio': 'avg'}).collect()[0][0]
avg_atual = df_atual.agg({'valor_premio': 'avg'}).collect()[0][0]

print(f'\nPreço médio apólice v0:    R$ {avg_v0:.2f}')
print(f'Preço médio apólice atual: R$ {avg_atual:.2f}')
print(f'Variação (reajuste 10%):   +{(avg_atual/avg_v0 - 1)*100:.1f}%')

Apólices na versão 0 (original): 30
Apólices na versão atual:         34

Preço médio apólice v0:    R$ 2995.00
Preço médio apólice atual: R$ 3091.91
Variação (reajuste 10%):   +3.2%


## 9. Contagem final — resumo após todos os DMLs

In [13]:
print('=== Contagem APÓS todos os DMLs ===')
for t in tabelas_dml:
    count = spark.read.format('delta').load(bronze_path(t)).count()
    print(f'  {t:12s}: {count:5d} registros')

print('\nDML concluído com sucesso!')

=== Contagem APÓS todos os DMLs ===
  cliente     :    66 registros
  apolice     :    34 registros
  sinistro    :    18 registros

DML concluído com sucesso!
